In [ ]:
!wget https://nasa-public-data.s3.amazonaws.com/plot3d_utilities/dataset-processed.zip
!unzip dataset-processed.zip

In [ ]:
!python preprocess_airfoil_dataset.py \
    --input_dir datasets/standard \
    --output_dir processed_output \
    --scaler_name standard

In [21]:
import numpy as np

train = np.load(
    "processed_output/standard/main/train.npz"
)

print(train.files)

['geometry_y', 'alpha', 'reynolds', 'ncrit', 'cl', 'cd', 'cdp', 'cm', 'cp']


In [22]:
X_train = np.hstack([
    train["geometry_y"],
    train["alpha"].reshape(-1,1),
    train["reynolds"].reshape(-1,1),
    train["ncrit"].reshape(-1,1)
])

y_train = np.stack([
    train["cl"],
    train["cd"]
], axis=1)

print(X_train.shape)
print(y_train.shape)

(457283, 201)
(457283, 2)


In [23]:
import torch
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

X_train = torch.tensor(X_train,dtype=torch.float32)
y_train = torch.tensor(y_train,dtype=torch.float32)

train_dataset = TensorDataset(X_train,y_train)

train_loader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True
)

In [24]:
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(201,256),
            nn.ReLU(),

            nn.Linear(256,128),
            nn.ReLU(),

            nn.Linear(128,64),
            nn.ReLU(),

            nn.Linear(64,2)
        )

    def forward(self,x):
        return self.net(x)

model = MLP().cuda()

In [25]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

best_loss = float("inf")

for epoch in range(20):

    model.train()

    running_loss = 0.0

    for x, y in train_loader:

        x = x.cuda()
        y = y.cuda()

        optimizer.zero_grad()

        pred = model(x)

        loss = criterion(pred, y)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    print(
        f"Epoch {epoch+1:02d} | Loss = {epoch_loss:.6f}"
    )

    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(
            model.state_dict(),
            "best_mlp.pt"
        )

print("Training complete.")
print("Best Loss:", best_loss)

Epoch 01 | Loss = 0.236240
Epoch 02 | Loss = 0.096432
Epoch 03 | Loss = 0.078418
Epoch 04 | Loss = 0.067863
Epoch 05 | Loss = 0.065269
Epoch 06 | Loss = 0.063042
Epoch 07 | Loss = 0.055296
Epoch 08 | Loss = 0.054088
Epoch 09 | Loss = 0.052134
Epoch 10 | Loss = 0.049339
Epoch 11 | Loss = 0.047035
Epoch 12 | Loss = 0.045613
Epoch 13 | Loss = 0.048422
Epoch 14 | Loss = 0.045413
Epoch 15 | Loss = 0.041500
Epoch 16 | Loss = 0.043006
Epoch 17 | Loss = 0.041013
Epoch 18 | Loss = 0.040179
Epoch 19 | Loss = 0.041270
Epoch 20 | Loss = 0.038848
Training complete.
Best Loss: 0.0388477950938196


In [27]:
# Cell 10: Evaluation

import numpy as np
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model.load_state_dict(
    torch.load("best_mlp.pt")
)

# Load validation data
val = np.load("processed_output/standard/main/val.npz")

# Build feature matrix
X_val = np.hstack([
    val["geometry_y"],
    val["alpha"].reshape(-1, 1),
    val["reynolds"].reshape(-1, 1),
    val["ncrit"].reshape(-1, 1)
])

# Targets: Cl and Cd
y_val = np.stack([
    val["cl"],
    val["cd"]
], axis=1)

# Convert to tensor
X_val_tensor = torch.tensor(
    X_val,
    dtype=torch.float32
).cuda()

# Inference
model.eval()

with torch.no_grad():
    pred = model(X_val_tensor)

pred = pred.cpu().numpy()

# Metrics
mae = mean_absolute_error(y_val, pred)
rmse = np.sqrt(mean_squared_error(y_val, pred))
r2 = r2_score(y_val, pred)

print(f"Overall MAE  : {mae:.6f}")
print(f"Overall RMSE : {rmse:.6f}")
print(f"Overall R²   : {r2:.6f}")

# Individual targets
cl_mae = mean_absolute_error(y_val[:, 0], pred[:, 0])
cl_rmse = np.sqrt(mean_squared_error(y_val[:, 0], pred[:, 0]))

cd_mae = mean_absolute_error(y_val[:, 1], pred[:, 1])
cd_rmse = np.sqrt(mean_squared_error(y_val[:, 1], pred[:, 1]))

print("\nLift Coefficient (Cl)")
print(f"MAE  : {cl_mae:.6f}")
print(f"RMSE : {cl_rmse:.6f}")

print("\nDrag Coefficient (Cd)")
print(f"MAE  : {cd_mae:.6f}")
print(f"RMSE : {cd_rmse:.6f}")

Overall MAE  : 0.113834
Overall RMSE : 0.200972
Overall R²   : 0.959401

Lift Coefficient (Cl)
MAE  : 0.080791
RMSE : 0.114037

Drag Coefficient (Cd)
MAE  : 0.146878
RMSE : 0.260342


Model saved as baseline_mlp.pt
